In [1]:
# Import library yang dibutuhkan
# pandas digunakan untuk membaca dan mengolah data tabel
# pathlib digunakan untuk mengatur path folder/file

import pandas as pd
from pathlib import Path

In [2]:
# Menentukan lokasi folder project
# BASE_DIR = folder utama project
# RAW_DIR = folder tempat data mentah disimpan
# PROCESSED_DIR = folder tempat data hasil preprocessing disimpan
# OIL_PRICE_DIR = folder khusus data harga minyak dunia

BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OIL_PRICE_DIR = RAW_DIR / "Oil-Price"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Oil Price directory:", OIL_PRICE_DIR)

Base directory: c:\Users\Project\skripsi\inflation-forecast-final
Oil Price directory: c:\Users\Project\skripsi\inflation-forecast-final\data\raw\Oil-Price


In [3]:
# Membaca dataset harga minyak Brent dari FRED
# Dataset ini masih berbentuk data harian

file_path = OIL_PRICE_DIR / "DCOILBRENTEU.csv"

oil_df = pd.read_csv(file_path)

oil_df.head()

,observation_date,DCOILBRENTEU
0,2010-01-04,79.05
1,2010-01-05,79.27
2,2010-01-06,80.14
3,2010-01-07,80.57
4,2010-01-08,80.06


In [4]:
# Mengecek ukuran dataset
# shape = (jumlah baris, jumlah kolom)

oil_df.shape


(4173, 2)

In [5]:
# Mengecek nama kolom dataset

oil_df.columns.tolist()

['observation_date', 'DCOILBRENTEU']

In [6]:
# Mengecek tipe data awal

oil_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4173 entries, 0 to 4172
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  4173 non-null   object 
 1   DCOILBRENTEU      4048 non-null   float64
dtypes: float64(1), object(1)
memory usage: 65.3+ KB


In [7]:
# Membuat salinan dataset agar data asli tetap aman

oil_price = oil_df.copy()

# Mengubah nama kolom agar lebih mudah dipahami
# observation_date -> date
# DCOILBRENTEU -> oil_price

oil_price = oil_price.rename(
    columns={
        "observation_date": "date",
        "DCOILBRENTEU": "oil_price"
    }
)

oil_price.head()

,date,oil_price
0,2010-01-04,79.05
1,2010-01-05,79.27
2,2010-01-06,80.14
3,2010-01-07,80.57
4,2010-01-08,80.06


In [8]:
# Mengubah kolom date menjadi datetime
# Mengubah oil_price menjadi numeric
# errors="coerce" digunakan agar nilai yang tidak bisa dibaca sebagai angka menjadi NaN

oil_price["date"] = pd.to_datetime(
    oil_price["date"],
    errors="coerce"
)

oil_price["oil_price"] = pd.to_numeric(
    oil_price["oil_price"],
    errors="coerce"
)

oil_price.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4173 entries, 0 to 4172
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       4173 non-null   datetime64[ns]
 1   oil_price  4048 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 65.3 KB


In [9]:
# Mengecek missing value pada data harian
# Data harga minyak harian bisa saja memiliki missing value karena hari libur atau data tidak tersedia

oil_price.isnull().sum()

date           0
oil_price    125
dtype: int64

In [10]:
# Mengecek periode awal dan akhir data harian

print(oil_price["date"].min())
print(oil_price["date"].max())

2010-01-04 00:00:00
2025-12-31 00:00:00


In [11]:
# Mengubah data harian menjadi data bulanan
# Karena dataset lain menggunakan frekuensi bulanan, oil price juga harus dibuat bulanan
# "MS" berarti Month Start, sehingga tanggal akan direpresentasikan sebagai awal bulan
# mean() digunakan untuk menghitung rata-rata harga minyak dalam satu bulan

oil_price_monthly = (
    oil_price
    .set_index("date")
    .resample("MS")["oil_price"]
    .mean()
    .reset_index()
)

oil_price_monthly.head()

,date,oil_price
0,2010-01-01,76.167368
1,2010-02-01,73.752105
2,2010-03-01,78.827391
3,2010-04-01,84.817619
4,2010-05-01,75.945500


In [12]:
# Memastikan periode data hanya dari Januari 2010 sampai Desember 2025
# Ini disesuaikan dengan dataset inflasi, BI Rate, dan exchange rate

oil_price_monthly = oil_price_monthly[
    (oil_price_monthly["date"] >= "2010-01-01") &
    (oil_price_monthly["date"] <= "2025-12-01")
].reset_index(drop=True)

oil_price_monthly

,date,oil_price
0,2010-01-01,76.167368
1,2010-02-01,73.752105
2,2010-03-01,78.827391
3,2010-04-01,84.817619
4,2010-05-01,75.945500
...,...,...
187,2025-08-01,67.870000
188,2025-09-01,67.985455
189,2025-10-01,64.543478
190,2025-11-01,63.797000


In [13]:
# Mengecek ukuran dataset
# Harusnya 192 baris dan 2 kolom
# 2010-2025 = 16 tahun
# 16 x 12 bulan = 192 observasi

oil_price_monthly.shape

(192, 2)

In [14]:
# Mengecek periode awal dan akhir data bulanan

print(oil_price_monthly["date"].min())
print(oil_price_monthly["date"].max())

2010-01-01 00:00:00
2025-12-01 00:00:00


In [15]:
# Mengecek missing value setelah data diubah menjadi bulanan
# Yang penting adalah tidak ada missing value pada level bulanan

oil_price_monthly.isnull().sum()

date         0
oil_price    0
dtype: int64

In [16]:
# Mengecek apakah ada tanggal yang duplikat

oil_price_monthly["date"].duplicated().sum()

np.int64(0)

In [17]:
# Menyimpan dataset oil price bulanan ke folder processed

output_path = PROCESSED_DIR / "oil_price_monthly.csv"

oil_price_monthly.to_csv(
    output_path,
    index=False
)

print(f"Dataset saved: {output_path}")

Dataset saved: c:\Users\Project\skripsi\inflation-forecast-final\data\processed\oil_price_monthly.csv


In [18]:
# Membaca kembali file yang sudah disimpan
# Tujuannya untuk memastikan file berhasil dibuat dan isinya tetap aman

saved_oil_price = pd.read_csv(
    PROCESSED_DIR / "oil_price_monthly.csv"
)

saved_oil_price.head()

,date,oil_price
0,2010-01-01,76.167368
1,2010-02-01,73.752105
2,2010-03-01,78.827391
3,2010-04-01,84.817619
4,2010-05-01,75.945500


In [19]:
# Mengecek ulang ukuran dataset setelah disimpan

saved_oil_price.shape

(192, 2)